# Kernel Initialization

```python

# Weights
nn.init.<method>(layer.weight)

# Bias
nn.init.<method>(layer.bias)


```

- Runs **once**, before training (in-place, hence the `_` suffix)
- Keeps the tensor's **dtype**
- Resamples weights with a chosen **mean** (0) and **std**
- **Gain** = correction factor for the activation function
- Goal: keep `Var(output) ≈ Var(input)` across layers (no exploding/vanishing signal)

Each method has its own variance formula, and each activation its own gain.
The general formula is:

$std = gain / √fan$        

$(Var(w) = gain² / fan)$

where `fan` is `n_in`, `n_out`, or `(n_in + n_out) / 2`.



# Xavier

**Use with:** sigmoid, tanh, linear



```python
nn.init.xavier_normal_(tensor, gain=1.0)
```


- `tensor` = `layer.weight`
- `gain` = correction factor for the activation (default **1.0**, so PyTorch assumes it if omitted)
- `std = gain · √(2 / (n_in + n_out))`
- `Var(w) = gain² · 2 / (n_in + n_out)`
- Mean = 0

Why `n_in + n_out`: a compromise between forward stability (`1/n_in`) and backward stability (`1/n_out`).

To get the gain, use:
```python
nn.init.calculate_gain("activation")
```

## Gain values

| Activation | Gain |
|---|---|
| linear / sigmoid | 1 |
| tanh | 5/3 |
| relu | √2 |

## Sanity check

```python
print(layer.weight.std())  # gain=1: ≈ √(2/384) ≈ 0.072
```

In [ ]:
import torch.nn as nn


g = nn.init.calculate_gain("tanh")
layer = nn.Linear(10, 10)
print(layer.weight)

print()
print()
print()

nn.init.xavier_normal_(layer.weight, gain = g)
print(layer.weight)


Parameter containing:
tensor([[ 0.2914, -0.2534, -0.1382,  0.1496,  0.1910,  0.1581, -0.1972, -0.1005,
          0.2174,  0.1185],
        [-0.1199, -0.0685, -0.2255,  0.2769, -0.0399,  0.1670,  0.0531,  0.2311,
          0.2578,  0.1889],
        [-0.1786, -0.1135, -0.1554,  0.0635,  0.1643,  0.1819,  0.2653, -0.0838,
         -0.1605,  0.1010],
        [ 0.2792,  0.2831,  0.0509, -0.3107, -0.3119,  0.1658, -0.2569,  0.1043,
         -0.0921, -0.0408],
        [ 0.0699,  0.0435, -0.2187, -0.0894,  0.0499, -0.3109, -0.2451,  0.0709,
         -0.1620,  0.2708],
        [-0.2153,  0.1025, -0.0961,  0.1940, -0.2509,  0.2122, -0.0408,  0.1480,
          0.1719,  0.1589],
        [ 0.0488, -0.2455, -0.1838,  0.1272, -0.2196, -0.2765,  0.2491,  0.2125,
         -0.0693,  0.1723],
        [ 0.0652,  0.2865,  0.0194, -0.3047, -0.2574, -0.0473, -0.0188,  0.2666,
         -0.1509,  0.2426],
        [-0.1885,  0.0022, -0.1079,  0.0880,  0.2879,  0.0484, -0.1928, -0.0138,
         -0.0783,  0.2808

# 2. He (Kaiming)

**Use with:** ReLU and its family (Leaky ReLU, GELU, SiLU)

```python
nn.init.kaiming_normal_(tensor, a=0, mode="fan_in", nonlinearity="leaky_relu")
```

- `tensor` = `layer.weight`
- `nonlinearity` = activation that follows the layer (PyTorch uses it to pick the gain)
- `a` = negative slope (only for `leaky_relu`)
- `mode` = `"fan_in"` (default, forward stability) or `"fan_out"` (backward stability)
- `std = gain / √fan`, with `fan = n_in` by default
- `Var(w) = gain² / n_in`
- Mean = 0

Why we must pass `nonlinearity`: the gain changes with the activation, so it is not fixed. Kaiming only receives the weight tensor and cannot see the activation layer.

He = Kaiming with `nonlinearity="relu"` (gain = √2, so `Var(w) = 2 / n_in`).

## Gain table

| Activation | Gain |
|---|---|
| linear / sigmoid | 1 |
| ReLU | √2 |
| Leaky ReLU (slope a) | √(2 / (1 + a²)) |
| tanh | 5/3 |

Get it with `nn.init.calculate_gain("relu")`

## Sanity check

```python
print(layer.weight.std())  # ReLU, n_in=256: √(2/256) ≈ 0.088
```

In [4]:
import torch.nn as nn


layer = nn.Linear(10, 10)
print(layer.weight)

print()
print()
print()

nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
print(layer.weight)


Parameter containing:
tensor([[ 0.2663, -0.1842,  0.1135,  0.1301, -0.1934, -0.1892,  0.0261,  0.0719,
          0.0320, -0.1157],
        [-0.2227,  0.1443,  0.0278, -0.1023,  0.2450,  0.1758,  0.1877,  0.2183,
          0.1434,  0.0781],
        [-0.0343, -0.2064, -0.1632,  0.2124, -0.0743,  0.2565, -0.1247,  0.1888,
         -0.1635,  0.1832],
        [ 0.2501,  0.0694, -0.1029,  0.0048, -0.0719,  0.1625, -0.1949, -0.0474,
         -0.0429, -0.2026],
        [ 0.1133, -0.0076,  0.0661, -0.2212, -0.1816,  0.0488,  0.1482, -0.2449,
          0.1786, -0.0100],
        [ 0.1069,  0.1556,  0.1857,  0.2611,  0.0103, -0.1964, -0.2415,  0.0732,
          0.1750,  0.2007],
        [ 0.3132, -0.0526, -0.0683, -0.2121,  0.0639, -0.0396,  0.1687,  0.0189,
          0.0025,  0.1676],
        [ 0.2101, -0.2589, -0.2818,  0.1968,  0.2363,  0.0856,  0.0422,  0.1485,
          0.2773,  0.1486],
        [ 0.0325, -0.2641,  0.0487,  0.1219,  0.0143,  0.0578, -0.1759, -0.2423,
          0.2150, -0.1961

# 3. Normal

**Use with:** Transformers (BERT, GPT-2, ViT, ...)

```python
nn.init.normal_(tensor, mean=0.0, std=1.0)
```

- `tensor` = `layer.weight`
- `mean` = 0
- `std` = fixed by you (~**0.02** in Transformers), **not** computed from `n_in`/`n_out`
- `Var(w) = std²` (here 0.0004)
- No `gain`, no `nonlinearity`: you pick the scale directly

Why a fixed std works: LayerNorm renormalizes the signal at every block, so the exact weight scale matters less than in nets without normalization.

## GPT-2 detail

Layers that write into the residual stream (attention output, MLP output) use a smaller std:

```python
std = 0.02 / math.sqrt(2 * n_layers)
nn.init.normal_(block.attn.proj.weight, mean=0.0, std=std)
nn.init.normal_(block.mlp.proj.weight, mean=0.0, std=std)
```
Each residual add increases variance, so scaling by `1/√(2N)` keeps it stable.

## Sanity check

```python
print(layer.weight.std())  # ≈ 0.02
```

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, mean=0.0, std=0.02)

model.apply(init_weights)

# 4. Truncated Normal

**Use with:** ViT, some Transformers

```python
nn.init.trunc_normal_(tensor, mean=0.0, std=1.0, a=-2.0, b=2.0)
```

- `tensor` = `layer.weight` (usually weights, not bias; bias is normally zeros)
- `mean` = 0
- `std` = fixed by you (~**0.02** in Transformers)
- `a`, `b` = **absolute** cutoffs (not in units of σ). Values outside `[a, b]` are resampled
- No `gain`, no `nonlinearity`

Why truncate: a normal has unbounded tails, so a rare weight can be far from 0. Truncating removes those extremes.

## Gotcha: a and b

The defaults `a=-2.0, b=2.0` are absolute. With `std=0.02`, they cut at ±100σ, which truncates nothing. To cut at ±2σ, set it yourself:

```python
nn.init.trunc_normal_(layer.weight, std=0.02, a=-0.04, b=0.04)  # ±2σ
```

## Sanity check

```python
print(layer.weight.std())  # slightly below 0.02
print(layer.weight.abs().max())  # ≤ 0.04
```

In [ ]:
import torch.nn as nn

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.trunc_normal_(m.weight, std=0.02, a=-0.04, b=0.04)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

model.apply(init_weights)

# 5. Zeros

**Use with:** biases (almost always), norm-layer shifts (beta)

```python
nn.init.zeros_(tensor)
```

- `tensor` = `layer.bias` (rarely `layer.weight`)
- Takes **only the tensor**. No `std`, `a`, `b`, `mean` or `gain`: nothing is sampled, every value is 0
- In-place, keeps dtype

Why zeros work for bias: the weights already break symmetry, and bias is added (not multiplied), so it does not affect the variance.

## Never use for weights

All-zero weights make every neuron identical: same output, same gradient, same update. Symmetry never breaks.

## Sanity check

```python
print(layer.bias.abs().max())  # 0
```

In [ ]:
import torch.nn as nn

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:      # bias=False gives None
            nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
        nn.init.ones_(m.weight)     # gamma
        nn.init.zeros_(m.bias)      # beta

model.apply(init_weights)

---

# Used in production: `.apply()`

**Use with:** applying one init rule to a whole model

`model.apply(fn)`

- `fn` = a function that receives **one module** `m`
- Walks **every submodule recursively** (children first, then the model itself) and calls `fn(m)` on each
- Returns the model; the init is in-place
- `fn` runs on **all** modules (Linear, ReLU, Sequential...), so filter with `isinstance`

## Practice: rule per layer type

```python
import torch
import torch.nn as nn

def init_weights(m):
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Embedding):
        nn.init.normal_(m.weight, std=0.02)

torch.manual_seed(42)        # reproducibility
model = MyModel()
model.apply(init_weights)    # general rule
```

## Exceptions after `apply`, THE HEAD IN THE LOGITS

```python
nn.init.normal_(model.head.weight, std=0.01)
nn.init.zeros_(model.head.bias)
```

## Inside the model

```python
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 10)
        self.apply(init_weights)   # runs at creation, can't be forgotten
```

## Production tips

- Set the **seed** before creating the model
- Call `apply` **before** `.to(device)` / DDP wrapping


In [ ]:
def init_weights(m):
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_normal_(m.weight, mode="fan_in", nonlinearity="relu")
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, (nn.BatchNorm2d, nn.LayerNorm)):
        nn.init.ones_(m.weight)    # gamma
        nn.init.zeros_(m.bias)     # beta

model = MyModel()
model.apply(init_weights)

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, 10)
        self.apply(init_weights)   # calls in the end of __init__